# S3 STARE-PODS Demo — AWS S3 + RDS Postgres

Cloud counterpart of `local_starepods_examples.ipynb`. Runs the same flow against real **AWS S3** (Parquet partitions) and a real **RDS Postgres** `PodsMetadata` table.

**Core workflow**
1. **Ingest GMI + SSMIS granules** → S3 Parquet + RDS metadata (two instruments so the overlap analytics in step 10 have something to compare)
2. **Find intersecting data** via STARE SIDs + RDS (bbox filter optional; default loads the full granule)
3. **Download intersecting Parquet partitions** from S3
4. **Reconstitute HDF5** (S1 + S2 scans)
5. **Structure comparison** — reconstituted vs original
6. **RDS metadata verification**

**Temporal features** (temporal-stare-pods issues 01–06)
7. **Temporal catalog** — every chunk carries `[t_start, t_end]` + podcode
8. **Period-filtered load** — data-level `[t_start, t_end]` overlap
9. **VCF temporal roll-up** — union range per pod, on the fly
10. **Multi-instrument overlap analytics** — GMI↔SSMIS rendezvous

> The S3/RDS temporal loaders read the **shared** `PodsMetadata` catalog — every ingest in the RDS table, not only this demo's granule (filtered by instrument). That is the production query surface, so the temporal counts reflect the whole catalog, unlike the local demo's fresh isolated SQLite.

**Requires** `starepandas/.config` with AWS + RDS credentials. Sample granules default to the in-repo GMI + SSMIS; override with `STAREPODS_SAMPLE_GRANULE` / `STAREPODS_SAMPLE_GRANULE_SSMIS`.

In [1]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_demo_and_construct_parallel",
#                       "-q"])

In [2]:
import os
import time
import h5py
from starepandas.demo_lib import StarePodsDemo
from starepandas.staredataframe import _ensure_rds_db_and_table

## Configuration

Edit these paths and parameters before running.

In [3]:
import starepandas

# AWS + RDS credentials. Resolved relative to the installed package so it works
# from any cwd (the .config lives next to the starepandas package).
CONFIG_PATH = os.path.join(os.path.dirname(os.path.abspath(starepandas.__file__)), ".config")

# Resolve the sample granule from the in-repo test-data dir so the notebook is
# safe to run anywhere (no dependency on an external sample directory). Override
# with the STAREPODS_SAMPLE_GRANULE env var to point at your own granule.
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(starepandas.__file__)))
GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5",
    ),
)

# GMI + SSMIS are a co-located pair (both 2025-01-01, concurrent orbits) whose
# ground tracks cross within ~3 min in 42 shared pods, so the overlap analytics
# (step 10) show genuine multi-instrument rendezvous.
SSMIS_GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE_SSMIS",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5",
    ),
)

# S3 root where Parquet partitions and RDS metadata for this demo live.
S3_PREFIX = "s3://zarrpods/gmi-demo-parquet"

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 4

# Bounding box filter — set to None to reconstitute the full granule
# (matching local_starepods_examples.ipynb), or e.g. (115, -30, 120, -25)
# to restrict to SW Australia / Perth.
BBOX = None   # full granule, no spatial filter — mirrors the local demo

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/gmi_s3_reconstituted.h5"

# Set to True to wipe S3_PREFIX (S3 objects + RDS metadata rows) before
# ingesting. Mirrors the local demo's CLEAN_BEFORE_RUN flag — prevents
# duplicate RDS rows on re-runs. Keep True unless you intentionally
# want to append more granules under the same prefix.
CLEAN_BEFORE_RUN = True

print(f"Granule  : {os.path.basename(GRANULE_FILE)}")
print(f"SSMIS    : {os.path.basename(SSMIS_GRANULE_FILE)}")
print(f"Datasets : {DATASETS}")
print(f"BBox     : {BBOX}  (None = full granule)")
print(f"S3 root  : {S3_PREFIX}")
print(f"Clean    : {CLEAN_BEFORE_RUN}")

Granule  : 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5
SSMIS    : 1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5
Datasets : ['GMI_S1', 'GMI_S2']
BBox     : None  (None = full granule)
S3 root  : s3://zarrpods/gmi-demo-parquet
Clean    : True


## Step 1 — Ingest GMI + SSMIS granules → S3 Parquet + RDS

In [4]:
%%time
demo = StarePodsDemo(aws_config_path=CONFIG_PATH)

s3_paths = demo.ingest_granules(
    data_path=GRANULE_FILE,
    instrument="GMI",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=CLEAN_BEFORE_RUN,
)
# Second instrument appends — clean_before_run=False so it does NOT wipe
# the GMI data just written to the same prefix.
ssmis_paths = demo.ingest_granules(
    data_path=SSMIS_GRANULE_FILE,
    instrument="SSMIS",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=False,
)
print(f"GMI  : stored {len(s3_paths)} dataset path(s)")
print(f"SSMIS: stored {len(ssmis_paths)} dataset path(s)")

# Granule basename — used as a substring filter on group_path. Note: as of
# the quaternary pod-code layout (2026-06-14) the S3 layout is FLAT and the
# granule basename is embedded in the chunk *filename*, bracketed by '-':
#   <S3_PREFIX>/<podcode>-<granule_basename>-<dataset>.parquet
# So the old startswith(S3_PREFIX + '/' + basename) scoping no longer matches.
# We use a substring match on the basename instead.
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]
granule_path_marker = f"-{granule_basename}-"   # matches the filename-embedded span

INFO:starepandas.ingest:clean_before_run=True → wiping s3://zarrpods/gmi-demo-parquet on S3 + RDS first
INFO:starepandas.ingest:clean_s3_prefix(s3://zarrpods/gmi-demo-parquet): deleted 1982 RDS row(s), 1982 S3 object(s)
INFO:starepandas.ingest:Ingesting GMI granules from /Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_pods_aws_parallel/tests/data/granules/1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5
INFO:starepandas.ingest:Found 1 GMI file(s)
INFO:starepandas.ingest:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5


Writing 265 Parquet partitions to S3...
  Progress: 50/265 partitions written...
  Progress: 100/265 partitions written...
  Progress: 150/265 partitions written...
  Progress: 200/265 partitions written...
  Progress: 250/265 partitions written...
✓ Inserted 265 metadata rows into RDS
✓ Finished writing 265 Parquet partitions to s3://zarrpods/gmi-demo-parquet
Writing 248 Parquet partitions to S3...
  Progress: 50/248 partitions written...
  Progress: 100/248 partitions written...
  Progress: 150/248 partitions written...
  Progress: 200/248 partitions written...


INFO:starepandas.ingest:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B.HDF5 → ['s3://zarrpods/gmi-demo-parquet', 's3://zarrpods/gmi-demo-parquet']
INFO:starepandas.ingest:Ingested 2 Parquet dataset(s)
INFO:starepandas.ingest:Ingesting SSMIS granules from /Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_pods_aws_parallel/tests/data/granules/1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5
INFO:starepandas.ingest:Found 1 SSMIS file(s)
INFO:starepandas.ingest:Processing 1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5


✓ Inserted 248 metadata rows into RDS
✓ Finished writing 248 Parquet partitions to s3://zarrpods/gmi-demo-parquet
Writing 370 Parquet partitions to S3...
  Progress: 50/370 partitions written...
  Progress: 100/370 partitions written...
  Progress: 150/370 partitions written...
  Progress: 200/370 partitions written...
  Progress: 250/370 partitions written...
  Progress: 300/370 partitions written...
  Progress: 350/370 partitions written...
✓ Inserted 370 metadata rows into RDS
✓ Finished writing 370 Parquet partitions to s3://zarrpods/gmi-demo-parquet
Writing 369 Parquet partitions to S3...
  Progress: 50/369 partitions written...
  Progress: 100/369 partitions written...
  Progress: 150/369 partitions written...
  Progress: 200/369 partitions written...
  Progress: 250/369 partitions written...
  Progress: 300/369 partitions written...
  Progress: 350/369 partitions written...
✓ Inserted 369 metadata rows into RDS
✓ Finished writing 369 Parquet partitions to s3://zarrpods/gmi-demo-

INFO:starepandas.ingest:✓ Stored 1C.F18.SSMIS.XCAL2021-V.20250101-S112813-E131004.078441.V07B.HDF5 → ['s3://zarrpods/gmi-demo-parquet', 's3://zarrpods/gmi-demo-parquet', 's3://zarrpods/gmi-demo-parquet', 's3://zarrpods/gmi-demo-parquet']
INFO:starepandas.ingest:Ingested 4 Parquet dataset(s)


✓ Inserted 367 metadata rows into RDS
✓ Finished writing 367 Parquet partitions to s3://zarrpods/gmi-demo-parquet
GMI  : stored 2 dataset path(s)
SSMIS: stored 4 dataset path(s)
CPU times: user 46.8 s, sys: 4.86 s, total: 51.6 s
Wall time: 4min 13s


## Step 2 — Find intersecting data via STARE SIDs

In [5]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
    intersecting = demo.find_intersecting_data(location_sids, instruments=["GMI"])
    # Scope to our granule. Substring match on the basename, which the flat
    # pod-code layout embeds in the chunk filename (bracketed by '-').
    if not intersecting.empty and "group_path" in intersecting.columns:
        intersecting = intersecting[
            intersecting["group_path"].str.contains(granule_path_marker, regex=False)
        ]
    print(f"Found {len(intersecting)} intersecting metadata row(s).")
else:
    location_sids = None
    intersecting = None
    print("BBOX is None — Step 4 will reconstitute the full granule directly.")

if intersecting is not None and not intersecting.empty:
    intersecting[["Dataset", "grouped_id", "group_path"]].head(8)


BBOX is None — Step 4 will reconstitute the full granule directly.


## Step 3 — Download intersecting Parquet partitions from S3

In [6]:
%%time
if intersecting is not None and not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting["Dataset"].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions to download — Step 4 will read S3 directly.")
    data_dict = {}

No intersecting partitions to download — Step 4 will read S3 directly.
CPU times: user 108 μs, sys: 61 μs, total: 169 μs
Wall time: 166 μs


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [7]:
%%time
# s3_prefix scope: with CLEAN_BEFORE_RUN=True the bucket only holds this
# granule's data, so passing the broad S3_PREFIX is correct and avoids the
# layout mismatch the old per-granule S3 prefix would create.
recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    s3_prefix=S3_PREFIX,
)
print(f"Written to: {recon_path}")


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S1' over full granule (no spatial filter)


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S2' over full granule (no spatial filter)


INFO:starepandas.demo_lib:✓ Reconstituted HDF5 written to /tmp/gmi_s3_reconstituted.h5


Written to: /tmp/gmi_s3_reconstituted.h5
CPU times: user 8.42 s, sys: 2.2 s, total: 10.6 s
Wall time: 1min 40s


## Step 5 — Structure comparison: reconstituted vs original

In [8]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_s3_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (2983, 221)          float32
  /S1/Longitude                                       (2983, 221)          float32
  /S1/Quality                                         (2983, 221)          int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (2983,)              float64
  /S1/SCstatus/SCaltitude                             (2983,)              float32
  /S1/SCstatus/SClatitude                             (2983,)              float32
  /S1/SCstatus/SClongitude                            (2983,)              float32
  /S1/SCstatus/SCorientation                          (2983,)              int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (2983,)              int8
  /S1/ScanTime/DayOfYear       

## Step 6 — RDS metadata verification

In [9]:
conn = _ensure_rds_db_and_table("StarePodsMetadata")
try:
    with conn.cursor() as cur:
        # Flat pod-code layout: the basename is embedded in the chunk
        # filename, so use a LIKE substring match (not a startswith prefix).
        cur.execute(
            'SELECT "Dataset", COUNT(*) '
            'FROM "PodsMetadata" '
            'WHERE "MetadataJson"->>%s LIKE %s '
            'GROUP BY "Dataset" ORDER BY "Dataset"',
            ("group_path", f"%{granule_path_marker}%"),
        )
        rows = cur.fetchall()
    print(f"RDS scope: group_path contains '{granule_path_marker}'")
    for ds, cnt in rows:
        print(f"  {ds}: {cnt} partition(s)")
finally:
    conn.close()


RDS scope: group_path contains '-1C.GPM.GMI.XCAL2016-C.20250101-S112952-E130304.061572.V07B-'
  GMI_S1: 265 partition(s)
  GMI_S2: 248 partition(s)


## Step 7 — Temporal catalog: every chunk carries `[t_start, t_end]` + podcode

`load_s3_temporal_catalog` returns the thin projection the analytics use (`podcode / Dataset / t_start / t_end`) from RDS — never the heavy `MetadataJson`. This reads the **shared** catalog filtered by instrument, so the counts include every GMI/SSMIS ingest in the table, not just this granule.

In [10]:
import pandas as pd
from starepandas.io.granules import (
    load_s3_metadata, load_s3_temporal_catalog, load_s3_vcf,
)
from starepandas.overlap import (
    rendezvous_events, overlap_matrix, overlap_pod_table, pair_drilldown,
)

catalog = pd.concat(
    [load_s3_temporal_catalog(dataset_prefix='GMI'),
     load_s3_temporal_catalog(dataset_prefix='SSMIS')],
    ignore_index=True,
)
print(f"Thin catalog (GMI+SSMIS, catalog-wide): {len(catalog)} chunks across "
      f"{catalog['Dataset'].nunique()} datasets")
display(catalog.groupby('Dataset').agg(
    chunks=('podcode', 'size'),
    first_start=('t_start', 'min'),
    last_end=('t_end', 'max'),
))
catalog.head(6)

Thin catalog (GMI+SSMIS, catalog-wide): 14738 chunks across 6 datasets


,chunks,first_start,last_end
Dataset,,,
GMI_S1,265,2025-01-01 11:29:53.002,2025-01-01 13:03:04.224
GMI_S2,248,2025-01-01 11:29:53.002,2025-01-01 13:03:04.224
SSMIS_S1,3571,2025-01-01 01:17:03.438,2025-01-01 19:57:30.586
SSMIS_S2,3567,2025-01-01 01:17:03.438,2025-01-01 19:57:30.586
SSMIS_S3,3529,2025-01-01 01:17:03.438,2025-01-01 19:57:30.586
SSMIS_S4,3558,2025-01-01 01:17:03.438,2025-01-01 19:57:30.586


,podcode,Dataset,t_start,t_end
0,q30220,GMI_S1,2025-01-01 11:29:53.002,2025-01-01 11:30:53.002
1,q33110,GMI_S1,2025-01-01 11:29:53.002,2025-01-01 11:31:38.002
2,q33300,GMI_S1,2025-01-01 11:29:53.002,2025-01-01 11:31:17.377
3,q30223,GMI_S1,2025-01-01 11:30:11.752,2025-01-01 11:30:51.127
4,q33303,GMI_S1,2025-01-01 11:30:19.252,2025-01-01 11:31:15.502
5,q30221,GMI_S1,2025-01-01 11:30:34.252,2025-01-01 11:30:53.002


## Step 8 — Period-filtered load

`load_s3_metadata(..., period=(start, end))` keeps only chunks whose **data-level** range `[t_start, t_end]` overlaps the period (the shared `_period_conditions` index-friendly rewrite — live EXPLAIN confirms a Bitmap Index Scan on `idx_pods_temporal`). A window bracketing the GMI window returns its chunks; a window days away returns none.

In [11]:
gmi = catalog[catalog['Dataset'].str.startswith('GMI')]
gmi_start, gmi_end = gmi['t_start'].min(), gmi['t_end'].max()
match_period = (gmi_start - pd.Timedelta(hours=1), gmi_end + pd.Timedelta(hours=1))
miss_period  = (gmi_start - pd.Timedelta(days=10), gmi_start - pd.Timedelta(days=9))

hit  = load_s3_metadata(dataset_prefix='GMI', period=match_period)
miss = load_s3_metadata(dataset_prefix='GMI', period=miss_period)

print(f"GMI catalog window : [{gmi_start}, {gmi_end}]")
print(f"bracketing period  -> {len(hit)} chunks")
print(f"9-10 days earlier  -> {len(miss)} chunks")

GMI catalog window : [2025-01-01 11:29:53.002000, 2025-01-01 13:03:04.224000]
bracketing period  -> 513 chunks
9-10 days earlier  -> 0 chunks


## Step 9 — VCF temporal roll-up

`load_s3_vcf(level, ...)` groups chunks by their level-`level` ancestor pod and returns each pod's union range `[min(t_start), max(t_end)]` + child count — the on-the-fly temporal hierarchy ("Virtual Collection File"), nothing materialized.

In [12]:
vcf = load_s3_vcf(1, dataset_prefix='GMI')
print(f"{len(vcf)} level-1 VCF nodes for GMI (one per octant subtree)")
vcf

14 level-1 VCF nodes for GMI (one per octant subtree)


,podcode,t_start,t_end,n_chunks,n_without_range
0,q12,2025-01-01 12:26:02.360,2025-01-01 12:39:09.856,55,0
1,q20,2025-01-01 12:37:07.982,2025-01-01 12:42:58.605,28,0
2,q22,2025-01-01 12:55:24.851,2025-01-01 12:58:32.350,8,0
3,q23,2025-01-01 12:41:37.980,2025-01-01 12:57:13.600,58,0
4,q30,2025-01-01 11:29:53.002,2025-01-01 13:03:04.224,26,0
5,q32,2025-01-01 11:36:15.500,2025-01-01 11:43:49.248,42,0
6,q33,2025-01-01 11:29:53.002,2025-01-01 13:03:04.224,46,0
7,q40,2025-01-01 11:40:15.499,2025-01-01 11:49:19.246,38,0
8,q60,2025-01-01 12:24:11.735,2025-01-01 12:28:54.859,20,0
9,q62,2025-01-01 12:09:37.989,2025-01-01 12:19:41.737,41,0


## Step 10 — Multi-instrument overlap analytics

`rendezvous_events` sweeps for passes simultaneously present in a pod within Δt; the matrix / pod-table / drill-downs aggregate that one events frame. Since the catalog-wide read (step 7) mixes every ingest in the shared RDS table, we scope the sweep to the **co-located pair's time window** (the `period` pushes into SQL) so the result is the pair's genuine rendezvous. Both granules are 2025-01-01 with concurrent orbits crossing within ~3 min — a pod where both have a chunk *and* their pass times fall within Δt is a spatial **and** temporal intersection.

In [13]:
pair_period = (pd.Timestamp('2025-01-01 11:00'), pd.Timestamp('2025-01-01 13:30'))
cat_pair = pd.concat(
    [load_s3_temporal_catalog(dataset_prefix='GMI', period=pair_period),
     load_s3_temporal_catalog(dataset_prefix='SSMIS', period=pair_period)],
    ignore_index=True,
)
dt = pd.Timedelta(minutes=15)
events = rendezvous_events(cat_pair, dt)
npods = events['podcode'].nunique() if not events.empty else 0
print(f"Rendezvous over the co-located GMI+SSMIS pair (dt={dt}): "
      f"{len(events)} events across {npods} shared pods")

print('\nInstrument x instrument matrix — pods where A & B rendezvous (slide 8):')
display(overlap_matrix(events))

print('Per-pod n-way combination counts (slide 9), first 10 pods:')
display(overlap_pod_table(events).head(10))

print('GMI-SSMIS pair drill-down (first 8 shared pods + crossing times):')
display(pair_drilldown(events, 'GMI', 'SSMIS').head(8))

Rendezvous over the co-located GMI+SSMIS pair (dt=0 days 00:15:00): 118 events across 42 shared pods

Instrument x instrument matrix — pods where A & B rendezvous (slide 8):


,GMI,SSMIS
GMI,0,42
SSMIS,42,0


Per-pod n-way combination counts (slide 9), first 10 pods:


n_instruments,2
podcode,
q22200,1
q22201,1
q22202,1
q22203,1
q23000,1
q23001,1
q23002,1
q23003,1
q23030,1


GMI-SSMIS pair drill-down (first 8 shared pods + crossing times):


,podcode,frequency,times
0,q22200,4,"[2025-01-01 12:57:00.713000, 2025-01-01 12:57:..."
1,q22201,2,"[2025-01-01 12:56:51.101000, 2025-01-01 12:56:..."
2,q22202,2,"[2025-01-01 12:55:24.851000, 2025-01-01 12:55:..."
3,q22203,4,"[2025-01-01 12:56:20.840000, 2025-01-01 12:56:..."
4,q23000,4,"[2025-01-01 12:57:00.713000, 2025-01-01 12:57:..."
5,q23001,4,"[2025-01-01 12:54:57.298000, 2025-01-01 12:54:..."
6,q23002,4,"[2025-01-01 12:57:31.093000, 2025-01-01 12:57:..."
7,q23003,4,"[2025-01-01 12:56:53.118000, 2025-01-01 12:56:..."
